In [147]:
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import datetime, time
import os
from dotenv import load_dotenv

from datetime import datetime
import ast
from itertools import chain
from typing import Optional
import matplotlib.pyplot as plt

import re
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [9]:

# status that represent active parts from site
PART_STATUS_ACTIVE = [1,2,4]


In [10]:

def establish_db_connection(server, database, username, password, driver):
    connection_string = (
        f"mssql+pyodbc://{username}:{password}@{server}/{database}"
        f"?driver={driver.replace(' ', '+')}"
    )
    engine = create_engine(connection_string)

    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT @@VERSION"))
            for row in result:
                print("Connected successfully. SQL Server version:")
                print(row[0])
            return engine
    except Exception as e:
        print("Connection failed:")
        print(e)
        return None

# Adjust to include error handling for the db connection method



In [11]:
def load_env():
    load_dotenv(dotenv_path="creds\\.env")


def SERVER_conn(input_site):

    load_env()

    # DB server
    site_server = os.getenv(input_site)
    
    
    paramz = {
        "site": (os.getenv('site_server')),
        "userName": os.getenv('USER_NAME'),
        "Password": os.getenv('PASSWORD_dev-test'),
        "Driver": os.getenv("ODBC_DRIVER")
    }

    db = os.getenv(input_site)

    server_conn = establish_db_connection(
        paramz["site"],
        db, 
        paramz["userName"],     
        paramz["Password"],
        paramz["Driver"])
        
    return server_conn


def db_request(query, server_conn_str):
    if server_conn_str is None:
        raise Exception("Database connection failed. Please check your credentials and connection settings.")

    # start_time = time.time()
    df = pd.read_sql(query, server_conn_str)

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"db_request must return a DataFrame, got {type(df)}")

    # end_time = time.time()
    # print(f"Query executed in {end_time - start_time:.2f} seconds")
    return df


In [12]:
# load_env()

# # DB server
# ast.literal_eval(os.getenv("DB_server_168"))[0]

In [13]:

# RO_all = "SELECT *  from Ops_tblRepairOrder where fldLastUpdated > '2020-01-1' AND fldStatus = 3 AND fldDivision IN (1)"
# query_all_requests = "SELECT *  from Ops_tblRequests where fldLastUpdated > '2020-01-1' AND fldAddWorkStatus IN (100, 300, 400)" 
# query_all_LabourLine = "SELECT *  from Ops_tblLabourLine where fldLastUpdated > '2020-01-1'"
# query_all_PartsLine = "SELECT *  from Ops_tblPartsLine where fldLastUpdated > '2020-01-1'"

# # More queries
# 
# 
# 

# get all F150 closed RO with relevant requests
RO_all = "SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_requests = "SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
query_all_PartsLine = "SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"
 

In [14]:
# search for op_codes_based_on_key_words

# select * from 
# Ops_tblOpCode2
# where fldDescription like ('%Water Pump%')

In [15]:


def pull_data_by_server(server_conn_str):
    # pull data for 
    RO_tbl = db_request(RO_all, server_conn_str)
    request_tbl = db_request(query_all_requests, server_conn_str)
    labourline_tbl = db_request(query_all_LabourLine, server_conn_str)
    partslines_tbl = db_request(query_all_PartsLine, server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl

def pull_data_by_server_with_args(server_conn_str, queries, modelName):
    # pull data for 

    # RO_tbl = server_conn_str.execute(queries["RO_tbl"], {"model": modelName}).fetchall()
    # request_tbl = server_conn_str.execute(queries["Req_tbl"], {"model": modelName}).fetchall()
    # labourline_tbl = server_conn_str.execute(queries["Labour_tbl"], {"model": modelName}).fetchall()
    # partslines_tbl = server_conn_str.execute(queries["Parts_tbl"], {"model": modelName}).fetchall()

    RO_tbl = db_request(queries["RO_tbl"], server_conn_str)
    request_tbl = db_request(queries["Req_tbl"], server_conn_str)
    labourline_tbl = db_request(queries["Labour_tbl"], server_conn_str)
    partslines_tbl = db_request(queries["Parts_tbl"], server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl

 

In [16]:

# # pull data for 
# RO_tbl_vw_174 = db_request(RO_all, vw_18_db)
# request_tbl_vw_174 = db_request(query_all_requests, vw_18_db)
# labourline_tbl_vw_174 = db_request(query_all_LabourLine, vw_18_db)
# partslines_tbl_vw_174 = db_request(query_all_PartsLine, vw_18_db)


In [17]:

# function to drop empty columns
def drop_empty_columns(df):
    df_cleaning = df.copy()
    # drop empty columns - must all empty
    df_cleaning = df_cleaning.dropna(axis=1, how='all')
    
    return df_cleaning
 

In [18]:
# function to filter columns
def filter_for_essential_columns(df, essential_cols):
    # df_selected = df[essential_cols].copy()   // this line is commented to grab all the columns for now, but can be re-enabled if we want to just focus on the essential columns
    df_selected = df.copy()
    return df_selected

In [19]:
# Defined essential columns for each table

essential_columns_request_tbl = ['fldId', 'fldWorkItemRef', 'fldSequence', 'fldDescription',
       'fldRequestCodeRef', 'fldRequestCode', 'fldRequestedTime', 'fldOrderNumber',
        'fldLastUpdated']


essential_cols_labourline_tbl = ['fldID', 'fldRequestRef', 'fldOpCodeRef',
       'fldActualHours', 'fldSoldHours', 'fldDescription',
       'fldAddedDate']

essential_cols_partlines_tbl = ['fldID', 'fldRequestRef', 'fldSequence', 'fldPartNumber', 'fldPartDesc',
       'fldRequested', 'fldShipped', 'fldOrderType', 'fldDateAdded', 'PC_PartDesc', 'fldPartsMasterRef']


essential_cols_RO_tbl = ['fldId', 'fldContactRef', 'fldVehicleRef', 'fldDateOpened',
       'fldDateClosed'
       ]
       
  

In [20]:

def clean_datset(df, tbl_type):
    df_dropped_empty_cols = drop_empty_columns(df)

    if tbl_type == "request":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_columns_request_tbl)
    
    elif tbl_type == "labourline":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_labourline_tbl)

    elif tbl_type == "partslines":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_partlines_tbl)

    elif tbl_type == "RO_tbl":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_RO_tbl)
        # remove
        
    return df_filtered
 

In [21]:

# request_tbl_vw_174 = clean_datset(request_tbl_vw_174, tbl_type="request")
# labor_tbl_vw_174 = clean_datset(labourline_tbl_vw_174, tbl_type="labourline")
# parts_tbl_vw_174 = clean_datset(partslines_tbl_vw_174, tbl_type="partslines")
# RO_tbl_vw_174 = clean_datset(RO_tbl_vw_174, tbl_type="RO_tbl")

#### Find a list of labour and parts for the following repair jobs 

- water pump 
- Timing belt
- Electrical - exterior lights


In [ ]:
def search_columns_for_keyword(df, keyword, column):
    if (column not in df.columns) or column=="":
        raise ValueError(f"Column '{column}' does not exist in the DataFrame.")
    filtered_df = df[df[column].str.contains(keyword, case=False, na=False)]
    return filtered_df

def get_top_ten_opcodes(df):
    top_ten = df["fldRequestCode"].value_counts().head(20)
    return top_ten

def search_request_by_opcode(df, opcode):
    search_result = df[df["fldRequestCode"]== opcode]
    
    return search_result

def search_request_by_list_of_opcodes(df, opcode_list):
    search_result = df[df["fldRequestCode"].isin(opcode_list)]
    
    return search_result



def part_items_metrics(parts_df):

    uniq_item_by_description = set(parts_df['fldPartDesc'].unique())
    metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])

    for desc in uniq_item_by_description:
        item_count = len(parts_df[parts_df['fldPartDesc'] == desc]) 
        total_units = parts_df[parts_df['fldPartDesc'] == desc]['fldRequested'].sum()
        uniq_partNumbers = parts_df[parts_df['fldPartDesc'] == desc]['fldPartNumber'].unique().tolist()
        new_row = {
                    'partDesc': desc, 
                    '#UniqParts': item_count, 
                    '#Qty': total_units,
                    'uniq_partNumbers': uniq_partNumbers
                    }
        
        # metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])
        
        new_row_df = pd.DataFrame([new_row]).reindex(columns=metrics.columns)
        metrics = pd.concat([metrics, new_row_df], ignore_index=True)
    return metrics




# def parts_summary(parts_tbl_df, total_req_count):
#     # Count occurrences of each unique part
#     # part_counts = parts_tbl_df['fldPartDesc'].value_counts()

#     part_counts = parts_tbl_df.groupby("fldPartDesc", as_index=False).agg(
#     count = ("fldPartNumber", "count"),
#     PartNum = ("fldPartNumber", lambda x: list(x.unique()))
#     ).sort_values("count", ascending=False)
#     part_counts = part_counts[~part_counts["fldPartDesc"].str.contains('ENV Fee|Core charge', case=False, regex=True)]


#     # Calculate percentage occurrence
#     part_counts["perc_occurence"] = round((part_counts['count'] / total_req_count * 100), 2)
    

#     # Sort for readability
#     metrics = metrics.sort_values(by='#perc_occurence', ascending=False)


#     # display(metrics)
#     return metrics



def parts_summary_v1(parts_tbl_df, total_req_count, similarity_threshold, ignore_words):
    """
    Summarizes parts occurrence and groups similar descriptions based on keyword similarity.
    
    Parameters:
    ----------
    parts_tbl_df : pd.DataFrame
        DataFrame containing part descriptions and request references.
    total_req_count : int
        Total number of requests for percentage calculation.
    similarity_threshold : float, optional (default=0.2)
        Jaccard similarity threshold for grouping descriptions.
    ignore_words : list of str, optional
        Words to ignore when determining similarity and forming combined names.
    
    Returns:
    -------
    pd.DataFrame
        DataFrame with combined part names and % occurrence.
    """
    
    if ignore_words is None:
        ignore_words = []
    
    # Normalize descriptions
    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    # Calculate initial metrics
    metrics = (
        parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
        .drop_duplicates()
        .groupby('fldPartDesc')
        .size()
        .reset_index(name='Count')
    )

    # Tokenize descriptions and remove ignored words
    metrics['Tokens'] = metrics['fldPartDesc'].apply(
        lambda x: set(word for word in re.split(r'\W+', x) if word and word not in [w.upper() for w in ignore_words])
    )

    # Group similar descriptions
    grouped = []
    visited = set()

    for i, row_i in metrics.iterrows():
        if i in visited:
            continue
        group = [i]
        for j, row_j in metrics.iterrows():
            if j in visited or i == j:
                continue
            # Jaccard similarity
            sim = len(row_i['Tokens'] & row_j['Tokens']) / len(row_i['Tokens'] | row_j['Tokens'])
            if sim >= similarity_threshold:
                group.append(j)
        visited.update(group)
        grouped.append(group)

    # Aggregate groups
    new_rows = []
    for group in grouped:
        part_names = metrics.loc[group, 'fldPartDesc'].tolist()
        counts = metrics.loc[group, 'Count'].sum()
        common_tokens = set.intersection(*metrics.loc[group, 'Tokens']) if len(group) > 1 else metrics.loc[group, 'Tokens'].iloc[0]
        common_name = " ".join(sorted(common_tokens)) if common_tokens else part_names[0]
        new_rows.append({'Part': common_name, 'Count': counts})

    # Create final DataFrame
    final_df = pd.DataFrame(new_rows)
    final_df['%Occurrence'] = (final_df['Count'] / total_req_count) * 100
    final_df['%Occurrence'] = final_df['%Occurrence'].round(2)
    final_df = final_df.sort_values(by='%Occurrence', ascending=False).reset_index(drop=True)
    final_df = final_df[["Part", "%Occurrence"]]
    return final_df



def parts_summary(parts_tbl_df):
    
    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    # Filter out non-relevant items based on keywords
    parts_tbl_df = parts_tbl_df[~parts_tbl_df["fldPartDesc"].str.contains("ENV FEE|CORE|Fluids", case=False, regex = True)]

    # count unique request references for each part description
    total_req_count = len(parts_tbl_df['fldRequestRef'].unique())



    # Agregate unique request counts by part description, count unique request references for each part description
    metrics = (parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
               .drop_duplicates()
               .groupby('fldPartDesc', as_index=False)
                .agg(
                    uniq_fldPartDesc_count= ("fldRequestRef","nunique")
                    )
                )
        
    # Calculate percentage occurrence for each part description
    metrics["freq_perc"] = (metrics["uniq_fldPartDesc_count"]/total_req_count * 100).round(2)

    # Create a list of unique part numbers for each part description
    partNumber_uniqueList = (parts_tbl_df.groupby('fldPartDesc',as_index=False)
                                .agg(PartNumbers = ('fldPartNumber', lambda x: list(pd.unique(x))))
                            )

    # Merge metrics with part numbers and sort by frequency percentage
    results = metrics.merge(partNumber_uniqueList, on = 'fldPartDesc')
    results = results.sort_values("freq_perc", ascending=False)

    #  Reset index and rename columns for better readability, also set index to start from 1 instead of 0
    results = results.reset_index(drop=True).set_axis(range(1, len(results) + 1))

    # # print sample size and unique opcodes for reference
    # print(f"Sample size: {total_req_count} ROs")
    
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    results.columns =["Part","Count", "frequency_%", "PartNumbers"]

    return results[["Part", "frequency_%", "PartNumbers"]]



def search_parts_and_labour_by_req_id(labor, parts, req_id):

    if "fldRequestRef" not in labor.columns or "fldRequestRef" not in parts.columns:
        raise ValueError("The required column 'fldRequestRef' does not exist in one of the DataFrames.")
    
    labor_result = labor[labor["fldRequestRef"]== req_id]
    parts_result = parts[parts["fldRequestRef"]== req_id]
        
    print(f"Labour items for Request ID {req_id}:")
    display(labor_result)
    print(f"Parts items for Request ID {req_id}:")
    display(part_items_metrics(parts_result))
    

In [23]:

def remove_invalid_opcodes(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(df)}")

    required_cols = ['fldFlatHours', 'fldTimeAllowed', 'fldCode']

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    # Apply filter
    return df[(df["fldFlatHours"] > 0) & (df["fldTimeAllowed"] > 0)]
    # return df



def get_valid_op_codes_by_keyword(db_conn, key_word: str) -> list:
    query = f"SELECT * FROM Ops_tblOpCode2 WHERE fldDescription LIKE '%{key_word}%'"
    search_result = db_request(query, db_conn)

    if search_result.empty:
        raise LookupError(f"No matching data for query: {query}")

    filter_results = remove_invalid_opcodes(search_result)

    if filter_results.empty:
        raise LookupError(f"No matching records found for keyword: {key_word}")

    return filter_results



def get_all_op_codes(db_conn) -> pd.DataFrame:
    """
    Retrieves all opcodes from the database.
    """
    query = "SELECT * FROM Ops_tblOpCode2"
    
    search_result = db_request(query, db_conn)

    return search_result


In [24]:

def clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl):
    RO_tbl_cleaned = clean_datset(RO_tbl, tbl_type="RO_tbl")
    request_tbl_cleaned = clean_datset(request_tbl, tbl_type="request")
    labourline_tbl_cleaned = clean_datset(labourline_tbl, tbl_type="labourline")
    partslines_tbl_cleaned = clean_datset(partslines_tbl, tbl_type="partslines")

    return RO_tbl_cleaned, request_tbl_cleaned, labourline_tbl_cleaned, partslines_tbl_cleaned


In [25]:

# Function to count the number of times a unique part item appears on a repair job

def parts_analysis(part_items, tracker_count_part_item_once_per_job, parts_summary_df):
    for index, row in part_items.iterrows():
            part_number = row["fldPartNumber"]
            part_desc = row["fldPartDesc"]

            # Count occurrence of each part item used on job             
            if (part_number in parts_summary_df["Part Number"].values) and (part_number not in tracker_count_part_item_once_per_job):
                parts_summary_df.loc[parts_summary_df["Part Number"] == part_number, "Occurrence_count"] += 1
                tracker_count_part_item_once_per_job.add(part_number)
            else:
                new_row = {
                    "Part Number": part_number,
                    "Part Description": part_desc,
                    "Occurrence_count": 1
                }
                parts_summary_df = pd.concat([parts_summary_df, pd.DataFrame([new_row])], ignore_index=True) 
                tracker_count_part_item_once_per_job.add(part_number)
                parts_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)


    return parts_summary_df, tracker_count_part_item_once_per_job
 

In [26]:


def filter_rows_by_keywords(df, column_name, keywords=[[], []], return_print=True):
    """
    Filters rows in a DataFrame where:
    - All keywords in the first sub-array must be present (AND logic).
    - At least one keyword in the second sub-array must be present (OR logic).
    
    Parameters:
    ----------
    df : pd.DataFrame
        The DataFrame to search.
    column_name : str
        The name of the column to search within.
    keywords : list of two lists
        keywords[0] = list of must-have keywords (AND condition)
        keywords[1] = list of optional keywords (at least one required)
    return_counts : bool, optional (default=True)
        If True, returns value counts of the filtered column.
        If False, returns the filtered DataFrame.
    
    Returns:
    -------
    pd.Series or pd.DataFrame
        Value counts of the filtered column or the filtered DataFrame.
    """
    
    must_have = keywords[0]
    optional = keywords[1]
    
    # Build regex for must-have keywords (AND logic using lookaheads)
    must_pattern = "".join(f"(?=.*{re.escape(word)})" for word in must_have)
    
    # Build regex for optional keywords (OR logic using |)
    optional_pattern = "|".join(re.escape(word) for word in optional)
    
    # Combine patterns: must-have AND (optional OR empty if none)
    if optional:
        pattern = f"{must_pattern}(?=.*(?:{optional_pattern}))"
    else:
        pattern = must_pattern
    
    # Apply filter
    mask = df[column_name].str.contains(pattern, case=False, regex=True, na=False)
    filtered_df = df[mask]

    # print(f"Sample size: {len(filtered_df)}")
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    key_columns= ['fldRequestCode', 'fldDescription']

    
    return filtered_df[key_columns] if return_print else filtered_df


In [27]:
def labour_items_analysis(labour_items, tracker_count_labour_item_once_par_job, labour_summary_df):
    for index, row in labour_items.iterrows():
            op_code = row["fldOpCodeRef"]
            labour_desc = row["fldDescription"]

            if op_code in labour_summary_df["fldOpCodeRef"].values and op_code not in tracker_count_labour_item_once_par_job:
                labour_summary_df.loc[labour_summary_df["fldOpCodeRef"] == op_code, "Occurrence_count"] += 1
                tracker_count_labour_item_once_par_job.add(op_code)
            else:
                new_row = {
                    "fldOpCodeRef": op_code,
                    "fldDescription": labour_desc,
                    "Occurrence_count": 1
                }
                labour_summary_df = pd.concat([labour_summary_df, pd.DataFrame([new_row])] , ignore_index=True )
                tracker_count_labour_item_once_par_job.add(op_code)
                labour_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)  
    return labour_summary_df, tracker_count_labour_item_once_par_job

In [28]:
def requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, request_summary_df):
    
    new_row = {
            "Request ID": req_id,
            "Description": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"].values[0],
            "fldRequestCode": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldRequestCode"].values[0],
            "#_PartItems": len(part_items),
            "#_LaborItems": len(labour_items)
            }

    request_summary_df = pd.concat([request_summary_df, pd.DataFrame([new_row])], ignore_index=True )
    request_summary_df.sort_values(by="#_PartItems", ascending=False, inplace=True)
    
    return request_summary_df

In [29]:


def run_analysis(labourline_tbl, partslines_tbl, request_tbl):


    filtered_requests_df = request_tbl
    
    # search from labour line and part line where fldRequestRef in filtered_requests_df['fldId']
    filtered_labour_df = labourline_tbl[labourline_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]
    filtered_parts_df = partslines_tbl[partslines_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]


    parts_summary_df = pd.DataFrame(columns=["Part Number", "Part Description", "Occurrence_count"])
    labour_summary_df = pd.DataFrame(columns=["fldOpCodeRef", "fldDescription", "Occurrence_count"])
    requestLine_summary_df = pd.DataFrame(columns=["Request ID", "Description","fldRequestCode", "#_PartItems", "#_LaborItems"])
 
    for items in filtered_requests_df['fldId'].values:
        req_id = items

        tracker_count_part_item_once_per_job = set()
        tracker_count_labour_item_once_par_job = set()
        

    # print(filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"])

        # search for all parts and labour lines for this req_id
        part_items = filtered_parts_df[filtered_parts_df["fldRequestRef"]== req_id]
        labour_items = filtered_labour_df[filtered_labour_df["fldRequestRef"]==req_id]

        requestLine_summary_df = requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, requestLine_summary_df)

        if not part_items.empty:
            parts_summary_df, tracker_count_part_item_once_per_job = parts_analysis(
                                                                                    part_items=part_items, 
                                                                                    tracker_count_part_item_once_per_job = tracker_count_part_item_once_per_job, 
                                                                                    parts_summary_df = parts_summary_df
                                                                                    )


        # if not labour_items.empty:
        #     labour_summary_df, tracker_count_labour_item_once_par_job = labour_items_analysis(
        #                                                                                 labour_items=labour_items, 
        #                                                                                 tracker_count_labour_item_once_par_job=tracker_count_labour_item_once_par_job, 
        #                                                                                 labour_summary_df=labour_summary_df
        #                                                                                 ) 
                                                           

    return filtered_parts_df

In [30]:
def plot_stats(requestLine_summary_df):

    parts_stats =  (
    requestLine_summary_df["#_PartItems"]
    .value_counts()
    .reset_index()
    .rename(columns={'index': '#_PartItems', '#_PartItems': '#Parts'})
    )

    labour_stats =  (
        requestLine_summary_df["#_LaborItems"]
        .value_counts()
        .reset_index()
        .rename(columns={'index': '#_LaborItems', '#_LaborItems': '#labour'})
    )

    # Sort for better visualization
    parts_stats = parts_stats.sort_values(by='#Parts').reset_index(drop=True)
    labour_stats = labour_stats.sort_values(by='#labour').reset_index(drop=True)


    # Compute stats for Parts
    mean_parts = parts_stats['#Parts'].mean()
    median_parts = parts_stats['#Parts'].median()
    mode_parts = parts_stats['#Parts'].mode()[0]

    # Compute stats for Labour
    mean_labour = labour_stats['#labour'].mean()
    median_labour = labour_stats['#labour'].median()
    mode_labour = labour_stats['#labour'].mode()[0]


    # Plot Parts line
    plt.plot(parts_stats['#Parts'], parts_stats['count'], color='blue', marker='o', label='Parts')

    # Plot Labour line
    plt.plot(labour_stats['#labour'], labour_stats['count'], color='green', marker='o', label='Labour')


    # Add reference lines for mean
    plt.axvline(mean_parts, color='blue', linestyle='--', alpha=0.5, label=f'Parts Mean: {mean_parts:.2f}')
    plt.axvline(mean_labour, color='green', linestyle='--', alpha=0.5, label=f'Labour Mean: {mean_labour:.2f}')


    # Annotate median and mode
    plt.text(parts_stats['#Parts'].max(), median_parts, f'Median: {median_parts}', color='blue')
    plt.text(parts_stats['#Parts'].max(), mode_parts, f'Mode: {mode_parts}', color='blue')
    plt.text(labour_stats['#labour'].max(), median_labour, f'Median: {median_labour}', color='green')
    plt.text(labour_stats['#labour'].max(), mode_labour, f'Mode: {mode_labour}', color='green')


    # Labels and title
    plt.xlabel('Item Count')
    plt.ylabel('Frequency')
    plt.title('Parts vs Labour Items with Summary Stats')
    plt.legend()
    plt.grid(True)
    plt.show()

In [255]:


def _normalize_partnumbers_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return [str(v).strip() for v in parsed if str(v).strip()]
            except Exception:
                pass
        return [p.strip() for p in s.split(",") if p.strip()]
    return [str(x).strip()] if str(x).strip() else []

def _dedupe_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _glob_to_regex(token: str, match_mode: str = "contains") -> str:
    """
    Convert a keyword with glob wildcards to regex.
      * => .*
      ? => .
    We escape everything else.
    """
    t = str(token).strip()
    if not t:
        return ""

    esc = re.escape(t)
    esc = esc.replace(r"\*", ".*").replace(r"\?", ".")
    if match_mode == "word":
        return rf"\b{esc}\b"
    return esc

def combine_parts_by_keyword_groups(
    df: pd.DataFrame,
    groups: dict,
    part_col: str = "Part",
    freq_col: str = "frequency_%",
    partnums_col: str = "PartNumbers",
    match_mode: str = "contains",  # "contains" or "word"
    dedupe_partnums: bool = True,
    combined_parts_col: str = "CombinedParts",
    matched_keywords_col: str = "MatchedKeywords",
) -> pd.DataFrame:
    """
    groups format:
      {
        "CANONICAL": [[include_any_patterns], [exclude_any_patterns]],
        ...
      }

    include_any_patterns: OR logic (at least one must match)
    exclude_any_patterns: NOT logic (none may match)

    Matching is case-insensitive (case=False) per your requirement.
    """

    required = {part_col, freq_col, partnums_col}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    if match_mode not in {"contains", "word"}:
        raise ValueError("match_mode must be 'contains' or 'word'")

    work = df.copy()
    work[freq_col] = pd.to_numeric(work[freq_col], errors="coerce").fillna(0)
    work[partnums_col] = work[partnums_col].apply(_normalize_partnumbers_cell)

    part_series = work[part_col].fillna("").astype(str)

    unassigned = pd.Series(True, index=work.index)
    matched_rows_out = []

    for canonical_name, rule in groups.items():
        if not isinstance(rule, (list, tuple)) or len(rule) != 2:
            raise ValueError(f"Group '{canonical_name}' must be [[include_any],[exclude_any]]")

        include_any, exclude_any = rule
        include_any = include_any or []
        exclude_any = exclude_any or []

        if len(include_any) == 0:
            continue

        # Build OR regex for includes
        include_patterns = [_glob_to_regex(k, match_mode) for k in include_any if str(k).strip()]
        include_patterns = [p for p in include_patterns if p]
        include_or = "(?:" + "|".join(include_patterns) + ")"

        include_mask = part_series.str.contains(include_or, case=False, regex=True, na=False)

        # Build OR regex for excludes (if any)
        if exclude_any:
            exclude_patterns = [_glob_to_regex(k, match_mode) for k in exclude_any if str(k).strip()]
            exclude_patterns = [p for p in exclude_patterns if p]
            if exclude_patterns:
                # exclude_or = "(" + "|".join(exclude_patterns) + ")"
                # exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

                
                exclude_or = "(?:" + "|".join(exclude_patterns) + ")"
                exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

            else:
                exclude_mask = pd.Series(False, index=work.index)
        else:
            exclude_mask = pd.Series(False, index=work.index)

        mask = include_mask & (~exclude_mask)

        group_idx = work.index[unassigned & mask]
        if len(group_idx) == 0:
            continue

        group_df = work.loc[group_idx]

        freq_sum = group_df[freq_col].sum()

        all_partnums = list(chain.from_iterable(group_df[partnums_col].tolist()))
        if dedupe_partnums:
            all_partnums = _dedupe_preserve_order(all_partnums)

        combined_parts = _dedupe_preserve_order(group_df[part_col].fillna("").astype(str).tolist())

        matched_rows_out.append({
            part_col: canonical_name,  # <-- group key goes into Part column
            freq_col: float(freq_sum),
            partnums_col: all_partnums,
            combined_parts_col: combined_parts,
            matched_keywords_col: {
                "include_any": include_any,
                "exclude_any": exclude_any
            }
        })

        unassigned.loc[group_idx] = False

    matched_df = pd.DataFrame(
        matched_rows_out,
        columns=[part_col, freq_col, partnums_col, combined_parts_col, matched_keywords_col]
    )


    remaining_df = work.loc[unassigned, [part_col, freq_col, partnums_col]].copy()
    remaining_df[combined_parts_col] = remaining_df[part_col].fillna("").astype(str).apply(lambda x: [x])
    remaining_df[matched_keywords_col] = [{"include_any": [], "exclude_any": []}] * len(remaining_df)

    final_df = pd.concat([matched_df, remaining_df], ignore_index=True)
    final_df[freq_col] = final_df[freq_col].round(2)
    final_df = final_df.sort_values(by=freq_col, ascending=False, kind="mergesort").reset_index(drop=True)

    # display(final_df)

    return final_df


Analysis for Ford Site 130

In [32]:

def combine_parts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Combine rows that share the same PartNumbers by:
      - summing frequency_%,
      - selecting longest Part string as the main Part,
      - concatenating all unique Part values into combined_parts,
      - keeping all other original columns,
      - returning df with no repeated PartNumbers,
      - sorted by frequency_% descending.
    """
    df = df.copy()

    # 1) Create hashable group key for PartNumbers
    df["_group_key"] = df["PartNumbers"].apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    )

    # 2) Ensure frequency_% is numeric
    df["frequency_%"] = df["frequency_%"].astype(float)

    # 3) Compute combined frequency
    df["_combined_freq"] = df.groupby("_group_key")["frequency_%"].transform("sum")

    # 4) Combined unique part names (joined string)
    df["combined_parts"] = df.groupby("_group_key")["Part"].transform(
        lambda s: ", ".join(sorted(set(map(str, s))))
    )

    # 5) Determine longest part string for each group
    longest_part_map = (
        df.groupby("_group_key")["Part"]
        .apply(lambda s: max(s, key=lambda x: len(str(x))))
    )
    df["_longest_part"] = df["_group_key"].map(longest_part_map)

    # 6) Drop duplicates (keep first row)
    df = df.drop_duplicates(subset="_group_key", keep="first")

    # 7) Replace Part and frequency_% with aggregated versions
    df["Part"] = df["_longest_part"]
    df["frequency_%"] = df["_combined_freq"]

    # 8) Cleanup helper columns
    df = df.drop(columns=["_combined_freq", "_group_key", "_longest_part"])

    # 9) Sort final df
    df = df.sort_values("frequency_%", ascending=False).reset_index(drop=True)

    return df




In [253]:
import ast
import pandas as pd

# =====================================================================
# 1) Load and normalize the master parts table ONCE (for performance)
# =====================================================================

# Path to your master parts CSV (adjust if needed)
PARTS_MASTER_PATH = "./data/parts_mastertbl.csv"

# Load the master parts DataFrame
parts_master_df = pd.read_csv(PARTS_MASTER_PATH)

# Strip whitespace from all column names to avoid matching issues
parts_master_df.columns = parts_master_df.columns.str.strip()

# Ensure the expected column exists
if "fldPartNumber" not in parts_master_df.columns:
    raise ValueError("Master DB missing required column: fldPartNumber")

# Create a set of normalized master part numbers for fast membership checks
# - Convert to string
# - Strip spaces
MASTER_PARTS = set(parts_master_df["fldPartNumber"].astype(str).str.strip())


# =====================================================================
# 2) Helper: convert a cell value into a Python list of values
# =====================================================================

def to_list(val):
    """
    Normalize a cell value into a Python list.

    Handles:
      - already-a-list values
      - NaN / missing values
      - stringified Python lists (e.g. "['123', '456']" or "[123, 456]")
      - comma-separated strings (e.g. "123, 456, 789")
      - scalars (int, float, str) by wrapping them into a list

    Returns:
      list
    """

    # Case 1: already a list → return as-is
    if isinstance(val, list):
        return val

    # Case 2: missing value (NaN, None, etc.) → treat as empty list
    if pd.isna(val):
        return []

    # Case 3: string values (could be list-like or comma-separated)
    if isinstance(val, str):
        s = val.strip()

        # Empty string → empty list
        if not s:
            return []

        # Try to interpret the string as a Python literal first, e.g. "['123', '456']"
        try:
            parsed = ast.literal_eval(s)

            # If it's already a list/tuple/set after parsing → convert to list
            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)

            # If it's a scalar (e.g. "123") → wrap in a list
            return [parsed]

        except (ValueError, SyntaxError):
            # Fallback: treat as comma-separated values, e.g. "123,456, 789"
            return [item.strip() for item in s.split(",") if item.strip()]

    # Case 4: any other scalar type (int, float, etc.) → wrap in a list
    return [val]


# =====================================================================
# 3) Helper: filter a list of part numbers against the master list
# =====================================================================

def filter_existing_parts(part_numbers, master_parts=MASTER_PARTS):
    """
    Given a list of part_numbers (any type), return only those
    that exist in the master parts set.

    Parameters:
      part_numbers : iterable of values (int/str/etc.)
      master_parts : set of normalized master part numbers (strings)

    Returns:
      list of strings: part numbers that exist in master
    """

    # Normalize incoming part numbers to string and strip
    part_numbers = [str(p).strip() for p in part_numbers]

    # Keep only those that exist in the master set
    valid_parts = [p for p in part_numbers if p in master_parts]

    return valid_parts

def get_OEM_and_None_OEM_from_list(part_numbers, master_parts_list, non_master_parts_list):
    """
    Given a list of part_numbers (any type), return only those
    that exist in the master parts set.

    Parameters:
      part_numbers : iterable of values (int/str/etc.)
      master_parts : set of normalized master part numbers (strings)

    Returns:
      list of strings: part numbers that exist in master
    """

    # get OEM parts and Non-OEM parts from the passed array
    OEM_parts = [p for p in part_numbers if p in master_parts_list]
    afterMrk_parts = [p for p in part_numbers if p in non_master_parts_list]

    return OEM_parts, afterMrk_parts


# =====================================================================
# 4) Row-wise function: compute intersection of part numbers across columns
# =====================================================================

def get_common_parts(row, partnumber_cols):
    """
    For a single row, compute the intersection of part numbers
    across the specified partnumber_cols.

    Assumes each column in partnumber_cols now contains a Python list
    of filtered part numbers (strings or scalars).

    Parameters:
      row            : a pandas Series (one row of the DataFrame)
      partnumber_cols: list of column names to consider

    Returns:
      - None if there is no common part number
      - set of strings containing the common part numbers otherwise
    """

    sets = []

    for col in partnumber_cols:
        # Convert the cell value to a list (in case it's not yet)
        lst = to_list(row[col])

        # Remove literal "nan" strings (from dirty data that may have "nan" as a string)
        cleaned_lst = [x for x in lst if str(x) != "nan"]

        if cleaned_lst:
            # Normalize all values to string for consistent set comparison
            value_set = set(map(str, cleaned_lst))
            sets.append(value_set)

    # If no valid lists were found in any column → no common parts
    if not sets:
        return None

    # Take the intersection across all sets
    common = set.intersection(*sets)

    # Return None if the intersection is empty, otherwise the set of common parts
    return common if common else None


# =====================================================================
# 5) Row-wise function: compute union (recommended parts) across columns
# =====================================================================

def get_recommended_parts(row, partnumber_cols):
    """
    For a single row, compute the union of part numbers
    across the specified partnumber_cols.

    Assumes each column in partnumber_cols now contains a Python list
    of filtered part numbers (strings).

    Parameters:
      row            : a pandas Series (one row of the DataFrame)
      partnumber_cols: list of column names to consider

    Returns:
      - None if there are no part numbers in any of these columns
      - list of unique strings (recommended parts) otherwise
    """

    # Use a set to ensure uniqueness across columns
    union_set = set()

    for col in partnumber_cols:
        # Convert the cell value to a list (in case it's not yet)
        lst = to_list(row[col])

        # Remove literal "nan" strings just in case
        cleaned_lst = [x for x in lst if str(x) != "nan"]

        # Normalize to string and add into the union set
        union_set.update(map(str, cleaned_lst))

    # If we ended up with no parts at all, return None
    if not union_set:
        return None

    # Return a sorted list for consistent ordering (optional)
    return sorted(union_set)


# =====================================================================
# 6) Main function: filter PartNumbers_* columns and add common & recommended
# =====================================================================

def combine_parts_by_partNumbers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame and:

      1. Finds all columns whose names contain 'PartNumbers'
      2. Converts their values into Python lists (using to_list)
      3. Filters each list so it only contains part numbers present in the master
      4. For each row:
         - computes the intersection of these filtered lists → 'common_partNumbers'
         - computes the union of these filtered lists       → 'recommended_parts'

    Returns:
      A new DataFrame (copy of the input) with:
        - PartNumbers_* columns cleaned and filtered
        - 'common_partNumbers' column
        - 'recommended_parts' column
    """

    # Work on a copy so we don't modify the caller's DataFrame in-place
    df = df.copy()

    # 1) Identify all columns that contain 'PartNumbers' in their name
    partnumber_cols = [col for col in df.columns if "PartNumbers" in col]

    # If there are no such columns, just return the DataFrame unchanged
    if not partnumber_cols:
        return df

    # 2) Filter each PartNumbers_* column against the master list
    for col in partnumber_cols:
        # For each cell:
        #   - convert to list
        #   - filter that list against the master parts set
        df[col] = df[col].apply(lambda v: filter_existing_parts(to_list(v)))

    # 3) Compute the intersection across the filtered lists row-by-row
    df["Common_Parts_Across_All"] = df.apply(
        get_common_parts, axis=1, partnumber_cols=partnumber_cols
    )

    # 4) Compute the union (recommended parts) across the filtered lists row-by-row
    df["Recommended_Parts"] = df.apply(
        get_recommended_parts, axis=1, partnumber_cols=partnumber_cols
    )

    return df


# =====================================================================
# 7) Example: compare results for two sites side-by-side
# =====================================================================

def compare_site_results_side_by_side(df_130, df_172, perc_thresh=10):
    """
    Compare part frequency results for two sites (e.g. 130 and 172) side-by-side.

    Pipeline:
      1. Call combine_parts(df) for each site (assumed to prep/aggregate parts)
      2. Filter each site to only include rows with frequency >= perc_thresh
      3. Determine subset size based on the smaller filtered set
      4. Merge both site DataFrames on 'Part' (outer join)
      5. Keep relevant columns and rename them for clarity
      6. Filter/clean/normalize all PartNumbers_* columns against the master
         and compute:
             - 'common_partNumbers'
             - 'recommended_parts'
      7. Sort by frequency in site 130 and keep top N based on subset size

    Parameters:
      df_130      : DataFrame for site 130
      df_172      : DataFrame for site 172
      perc_thresh : frequency threshold (inclusive) as a percentage (int)

    Returns:
      DataFrame:
        Columns:
          - Part
          - Freq_130
          - Freq_172
          - PartNumbers_130 (filtered lists)
          - PartNumbers_172 (filtered lists)
          - common_partNumbers (set or None)
          - recommended_parts (list or None)
        Rows:
          Top N parts sorted by Freq_130, where N is the smaller
          of the two thresholded subsets.
    """

    # -----------------------------------------------------------------
    # 1) Preprocess each site's DataFrame
    # -----------------------------------------------------------------
    # Assumes combine_parts(df) is defined elsewhere and:
    #   - groups/aggregates by Part
    #   - creates frequency_% column
    #   - and something like PartNumbers column
    df_130 = combine_parts(df_130)
    df_172 = combine_parts(df_172)

    # -----------------------------------------------------------------
    # 2) Filter by frequency threshold for each site
    # -----------------------------------------------------------------
    select_top_n_130 = df_130[df_130["frequency_%"].astype(int) >= perc_thresh]
    select_top_n_172 = df_172[df_172["frequency_%"].astype(int) >= perc_thresh]

    # -----------------------------------------------------------------
    # 3) Determine subset size: use the smaller of the two filtered sets
    # -----------------------------------------------------------------
    subset_size = max(len(select_top_n_130), len(select_top_n_172))

    # -----------------------------------------------------------------
    # 4) Merge site DataFrames on 'Part' using outer join
    # -----------------------------------------------------------------
    merged_results = df_130.merge(df_172, on="Part", how="outer")

    # Keep only the essential columns for comparison
    result_with_select_columns = merged_results[
        ["Part", "frequency_%_x", "frequency_%_y", "PartNumbers_x", "PartNumbers_y"]
    ]

    # Rename columns to more readable names
    result_with_select_columns.columns = [
        "Part",
        "Freq_130",
        "Freq_172",
        "PartNumbers_130",
        "PartNumbers_172",
    ]

    # -----------------------------------------------------------------
    # 5) Filter PartNumbers_* against master, and compute common & recommended
    # -----------------------------------------------------------------
    result_with_select_columns = combine_parts_by_partNumbers(result_with_select_columns)

    # -----------------------------------------------------------------
    # 6) Sort by frequency for site 130 and keep top subset_size rows
    # -----------------------------------------------------------------
    result_with_select_columns = (
        result_with_select_columns
        .sort_values("Freq_130", ascending=False)
        .reset_index(drop=True)
    )
    result_with_select_columns = result_with_select_columns[["Part","Recommended_Parts", "Common_Parts_Across_All" ,  "PartNumbers_130", "PartNumbers_172"]]
    result_with_select_columns = result_with_select_columns.head(subset_size)

    return result_with_select_columns



from functools import reduce

def merge_dict_of_dfs(data_dict, on="part", how="outer"):
    """
    Merge a dict of DataFrames on a common column.

    Parameters
    ----------
    data_dict : dict[str, pd.DataFrame]
        Dict mapping a key (e.g. site id) to a DataFrame.
    on : str
        Name of the common column to merge on (e.g. 'part').
    how : str
        Type of merge to perform (e.g. 'outer', 'inner', 'left', 'right').

    Returns
    -------
    pd.DataFrame
        Merged DataFrame with columns renamed as <key>_<original_col>
        for all columns except the join column.
    """

    # 1. Rename columns in each df to include the dict key as a prefix
    renamed_dfs = []
    for key, df in data_dict.items():
        # Build a rename mapping for all columns except the join column
        col_map = {
            col: f"{key}_{col}"
            for col in df.columns
            if col != on
        }
        df_renamed = df.rename(columns=col_map)
        renamed_dfs.append(df_renamed)

    # 2. Reduce-merge all DataFrames on the common column
    merged = reduce(
        lambda left, right: pd.merge(left, right, on=on, how=how),
        renamed_dfs
    )

    return merged

def select_essential_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame containing only 'essential' columns.
    Essential columns are those whose names contain any of the
    keywords: 'frequency', 'PartNumbers' (case-insensitive).

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame with only the matching columns.
    """
    key_words = ["frequency", "PartNumbers"]

    # Normalize column names to strings in case some are not
    cols = df.columns.astype(str)

    # Build a boolean mask: True if any keyword is in the column name
    mask = cols.str.lower().map(
        lambda col: any(kw.lower() in col for kw in key_words)
    )

    selected_cols = list(cols[mask])
    
    # Always keep 'Part' if it exists in the original df
    if "Part" in df.columns and "Part" not in selected_cols:
        selected_cols.append("Part")

    final_df = df.loc[:, selected_cols] 
    
    return final_df

def get_frequency_columns(df):
    
    freq_key_word = ["frequency"]
    # Normalize column names to strings in case some are not
    cols = df.columns.astype(str)

    # Build a boolean mask: True if any keyword is in the column name
    mask = cols.str.lower().map(
        lambda col: any(kw.lower() in col for kw in freq_key_word)
    )

    selected_cols = list(cols[mask])
    return selected_cols

def get_max_freq_among_all(df):
    
    selected_cols = get_frequency_columns(df)

    # loop through the columns, 
    col_with_highest_freq = selected_cols[0]
    highest_freq = 0

    for col in selected_cols:
        
        col_with_highest_freq = col if df[col].max() > highest_freq else None
        highest_freq = df[col].max()

    
    return col_with_highest_freq


def add_average_column(df: pd.DataFrame, new_col_name: str = "Avg_Freq_%") -> pd.DataFrame:
    selected_columns = get_frequency_columns(df)
    
    missing = [col for col in selected_columns if col not in df.columns]
    if missing:
        raise ValueError(f"The following columns are missing from df: {missing}")

    # Convert to numeric, coerce errors to NaN, then take mean
    numeric_block = df[selected_columns].apply(pd.to_numeric, errors="coerce")

    df[new_col_name] = numeric_block.mean(axis=1).round(1)

    return df


def safe_parse_list(value) -> Optional[list]:
        """
        Safely parse a string that represents a Python list.
        Returns a list or None if parsing fails or value is empty.
        """
        if pd.isna(value):
            return None

        if isinstance(value, list):
            # Already a list
            return value

        if isinstance(value, str):
            text = value.strip()
            if text == "" or text == "[]":
                return None
            try:
                parsed = ast.literal_eval(text)
                # Ensure we return a list or None, not anything else
                if isinstance(parsed, list):
                    return parsed if parsed else None
                else:
                    # If it is a single string or something else, wrap in a list
                    return [str(parsed)]
            except (SyntaxError, ValueError):
                # If parsing fails, treat as single value
                return [text]

        # Fallback: treat as single value
        return [str(value)]



def process_recommended_parts(input_df: pd.DataFrame,
                              target_col: str = "Recommended_Parts") -> pd.DataFrame:
    """
    Process the target column containing stringified lists of part numbers and
    split them into OEM_Part and non_OEM_Parts based on is_from_main().

    Assumes there is a function is_from_main(part: str) -> bool available.
    """
    df = input_df.copy()
    # Ensure target column exists
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in DataFrame.")

    # Create columns if they do not exist
    if "OEM_Part" not in df.columns:
        df["OEM_Part"] = None
    if "non_OEM_Parts" not in df.columns:
        df["non_OEM_Parts"] = None

    # Process each row
    for idx, row in df.iterrows():
        raw_value = row[target_col]
        # print(raw_value)
        # break

        # Convert stringified list to an actual list
        parts_list = to_list(raw_value)
        # break
        main_source = []
        non_main_source = []

        
        if not parts_list:
            df.at[idx, "OEM_Part"] = None
            df.at[idx, "non_OEM_Parts"] = None
            continue

        # get OEM and Non_OEM parts
        master_parts_list = parts_master_df[parts_master_df["fldPartsMasterRef"] == 1]["fldPartNumber"].to_list()
        non_master_parts_list = parts_master_df[parts_master_df["fldPartsMasterRef"] != 1]["fldPartNumber"].to_list()

        # print(f"Passed list {parts_list}")
        # print(f"Master: {master_parts_list}")
        # print(f"NonMaster: {non_master_parts_list}")
        

        OEM_parts, non_OEM_parts = get_OEM_and_None_OEM_from_list(parts_list, master_parts_list, non_master_parts_list)
        # print(f"OEM: {OEM_parts}")
        # Convert lists to comma-separated strings or None if empty
        
        df.at[idx, "OEM_Part"] = ", ".join(OEM_parts) if OEM_parts else None
        df.at[idx, "non_OEM_Parts"] = ", ".join(non_OEM_parts) if non_OEM_parts else None

    return df


def compare_recommended_partLists_side_by_side(list_of_sites, perc_thresh=10):
    
    selected_data_dict ={}

    for key, df in list_of_sites.items():

        combined_df = combine_parts(df) # combined same parts
        select_common_parts_by_set_freq = combined_df[combined_df["frequency_%"].astype(int) >= perc_thresh]

        selected_data_dict[key] = select_common_parts_by_set_freq
        
        # print(f"{selected_data_dict.keys()}")
        
        # display(select_common_parts_by_set_freq.head())        

    # merge all dfs
    merged_df_with_freq_summary = merge_dict_of_dfs(selected_data_dict, on="Part", how="outer")

    # display(merged_df_with_freq_summary)
    filtered_freq_summary = select_essential_columns(merged_df_with_freq_summary)

    # print(type(filtered_freq_summary))
    # display(merged_df_with_freq_summary.head())
    filtered_freq_summary_combined_part_numbers = combine_parts_by_partNumbers(filtered_freq_summary)
    

    df_with_final_freq = add_average_column(filtered_freq_summary_combined_part_numbers)

    df_split_OEM_and_none = process_recommended_parts(df_with_final_freq)
    # col_to_sort_by = get_max_freq_among_all(filtered_freq_summary_combined_part_numbers)

    sorted_summary_results = df_split_OEM_and_none.sort_values(by = "Avg_Freq_%", ascending=False)

    first_cols = ["Part", "Avg_Freq_%", "Recommended_Parts","OEM_Part","non_OEM_Parts", "Common_Parts_Across_All"]
    other_cols = [c for c in sorted_summary_results.columns if c not in first_cols]
    final_summary_df = sorted_summary_results.loc[:, first_cols + other_cols]
    final_summary_df = final_summary_df.reset_index(drop=True)
    final_summary_df.index = final_summary_df.index + 1 
    return final_summary_df
    
 
    


In [251]:

# data = {
#     "130": results130_water_pump,
#     "174": results172_water_pump,
#     "114": results114_water_pump,
#     "129": results129_water_pump,
#     "168": results168_water_pump
# }

# compare_recommended_partLists_side_by_side(data)
 

In [35]:
# a = pd.DataFrame({"fldPartNumber":[1,2,3,4,5,6,7]})

# display(a)



# check_list = [1,8, 10]

# filter_existing_parts(check_list, a)



In [ ]:


def execute_model(request_tbl_df, search_key_words, labour_line_df, parts_line_df,
                  similarity_threshold, ignore_words, groups):

    # Normalize input: allow "Water Pump" or [["water","pump"], []]
    if isinstance(search_key_words, str):
        search_key_words = [[search_key_words], []]

    requests_filtered = filter_rows_by_keywords(
        request_tbl_df, "fldDescription", search_key_words, return_print = False
    )

    # Vectorized: filter parts only once
    req_ids = requests_filtered["fldId"].unique()
    filtered_parts_df = parts_line_df[parts_line_df["fldRequestRef"].isin(req_ids)].copy()
    parts_list = parts_summary(filtered_parts_df)

    
    # print("-----------------------------------------------------------------------------")
    final_df = combine_parts_by_keyword_groups(
        df=parts_list,
        groups=groups,
        part_col="Part",
        freq_col="frequency_%",       # <-- use your actual frequency column name
        partnums_col="PartNumbers",
        match_mode="contains"         # or "word"
    ).reset_index(drop=True)
    
    # print(f"")

    # display(final_df)
    return final_df



Illustration for Site 172

In [37]:


# db_server_172 = "DB_server_172"
# # key_wrd = "Water Pump Replace"

# server_conn_db_172 = SERVER_conn(db_server_172)

# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = pull_data_by_server(server_conn_db_172)
# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = clean_data(RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172)



In [38]:


# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = pull_data_by_server(server_conn_db_130)
# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = clean_data(RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130)

 

In [39]:

# # Repair 1: Water pump replace - Site 172

# # search_key_words_water_pump_172 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_172 = [["water", "pump"], []]

# print("Water pump - Site 172")
# resutlts_water_pump_site_172 = execute_model(request_tbl_db_172, search_key_words_water_pump_172, labourline_tbl_db_172, partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - ", "kit", "ASY", "Rep"])

# # Repair 1: Water pump replace - Site 130

# # search_key_words_water_pump_130 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_130 = [["water", "pump"], []]
# print("Water pump - Site 130")
# resutlts_water_pump_site_130 = execute_model(request_tbl_db_130, search_key_words_water_pump_130, labourline_tbl_db_130, partslines_tbl_db_130, similarity_threshold= 0.6, ignore_words=[" - ", "kit", "ASY", "Rep"])


# # Repair 2 : Catalytic Converter Replace - Site 172

# search_key_words_catalytic_replace_172 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_172, "fldDescription", search_key_words_battery_replace_172, False)

# print("Catalytic Converter - Site 172")
# resutlts_Catalytic_Converter_site_172 = execute_model(request_tbl_df= request_tbl_db_172, search_key_words= search_key_words_catalytic_replace_172, labour_line_df= labourline_tbl_db_172, parts_line_df= partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])
 

# # Repair 2 : Catalytic Converter Replace - Site 130

# search_key_words_catalytic_replace_130 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_130, "fldDescription", search_key_words_battery_replace_130, False)

# print("Catalytic Converter - Site 130")
# resutlts_Catalytic_Converter_site_130 = execute_model(request_tbl_df= request_tbl_db_130, search_key_words= search_key_words_catalytic_replace_130, labour_line_df= labourline_tbl_db_130, parts_line_df= partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "])
 


# # Repair 3 : Power Steering - Site 172

# # search_key_words_catalytic_replace_172 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_172 = [["Power steering"], []]
# print("Power Steering - Site 172")
# resutlts_Power_Steering_site_172 = execute_model(request_tbl_df = request_tbl_db_172, search_key_words = search_key_words_catalytic_replace_172, labour_line_df = labourline_tbl_db_172, parts_line_df = partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])


 
# # Repair 3 : Power Steering - Site 130

# # search_key_words_catalytic_replace_130 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_130 = [["Power steering"], []]
# print("Power Steering - Site 130")
# resutlts_Power_Steering_site_130 = execute_model(request_tbl_df = request_tbl_db_130, search_key_words = search_key_words_catalytic_replace_130, labour_line_df = labourline_tbl_db_130, parts_line_df = partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "]) 
 

In [40]:

# compare_water_pump = resutlts_water_pump_site_172.merge(resutlts_water_pump_site_130, on="Part")
# compare_water_pump


# # print(type(resutlts_water_pump_site_172))

In [41]:
# Construct queries 


def is_validModel(server_conn, model):
    query = f" SELECT * FROM Veh_tblModel WHERE fldName = '{model}' AND fldInActive = 0"

    retults = db_request(query, server_conn)
    return len(retults)

def query_constructor(model, server):
    load_env()

    Queries= dict()
    if server == "DB_server_168":
        division = '5'
    else :
        division = '1'

    RO_all = f"SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN ({division}) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_requests = f"SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN ({division}) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
    
    query_all_PartsLine = f"SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded, PC.fldPartDescription as PC_PartDesc, PC.fldPartsMasterRef, PC.fldPartDescription, PC.fldStatus, PC.fldClassRef, PC.fldManufacturerRef FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId INNER JOIN Parts_tblPartsCurrent PC WITH(NOLOCK) on PL.fldPartNumber = PC.fldPartNumber WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN ({division}) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN ({division}) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    Queries["RO_tbl"] = RO_all
    Queries["Req_tbl"] = query_all_requests
    Queries["Parts_tbl"] = query_all_PartsLine
    Queries["Labour_tbl"] = query_all_LabourLine

    return Queries

In [42]:


def data_pull(modelName, db_server):
    
    # key_wrd = "Replace Water Pump"
    server_conn = SERVER_conn(db_server)
    
    if not is_validModel(server_conn, modelName):
        raise ValueError("Provided Model Name does not exists")
    queries = query_constructor(modelName, db_server) # pass in model name and 
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = pull_data_by_server_with_args(server_conn, queries, modelName)
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl)
    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl


def save_data(directoryPath : str, df, siteName):
    # df.to_csv('dat/site')
    return 


def run_model(key_wrd, request_tbl, labourline_tbl, partslines_tbl):
    execute_model(request_tbl_df = request_tbl, search_key_words = key_wrd, labour_line_df = labourline_tbl, parts_line_df = partslines_tbl, similarity_threshold=0.7, ignore_words=[" - "]) 




    

In [43]:

def dataset_info(RO_tbl, req_tbl, labour_tbl, parts_tbl, site):

    print(f"Sample Size: Site {site}")

    print(f"RO_tbl: {len(RO_tbl)}")
    print(f"Req_tbl: {len(req_tbl)}")
    print(f"LabourLines_tbl: {len(labour_tbl)}")
    print(f"PartLines_tbl: {len(parts_tbl)}")


In [44]:
# partslines_tbl_130_f150.head()

In [45]:

# WATER PUMP", "PUMP ASY", "PUMP ASY - WA*", "KIT - WATER"
#  Keywords for grouping parts into categories (example provided, can be expanded as needed)
groups= {
    "WATER PUMP (KIT/ASY)": [["WATER", "PUMP"], ["Gasket","Pulley","HOSE","COVER","OIL","CONNECTION","WATER BYP","FUE","TUBE","ADAPTOR","WASHER"]],
    "MC YELLOW COOLANT":[["COOLANT"], []],
    "GASKET - WATER PUMP ":[["GASKET"],[]],
    "POWER STEERING (ALL)": [["POWER STEERING", "P/S", "STEERING PUMP"],[]],
    "CATALYTIC CONVERTER (ALL)": [["CATALYTIC", "CAT CONVERTER", "CONVERTER"],[]]
}

In [46]:
# key_wrd = "Water Pump"
# results_tbl_130_f150 = execute_model(request_tbl_df = request_tbl_130_f150, search_key_words = key_wrd, labour_line_df = labourline_tbl_130_f150, parts_line_df = partslines_tbl_130_f150, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

In [47]:

def save_data_locally(data, base_path, server_name):
    # Build final directory: base_path/server_name
    final_path = os.path.join(base_path, server_name)
    
    # Create directory if it doesn't exist
    os.makedirs(final_path, exist_ok=True)
    
    # Basic validation
    if not isinstance(data, dict):
        raise ValueError("`data` must be a dict of pandas DataFrames.")
    
    # Save each DataFrame
    for key, df in data.items():
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Value for key '{key}' is {type(df)}, expected pandas.DataFrame")

        file_path = os.path.join(final_path, f"{key}.csv")
        print(f"Saving {key} -> {file_path}")  # DEBUG print
        df.to_csv(file_path, index=False)

    print(f"Done. Saved {len(data)} file(s) into: {final_path}")


# # ---- TEST IT ----
# data = {
#     "RO_tbl": pd.DataFrame({"a": [1, 2], "b": [3, 4]}),
#     "Req_tbl": pd.DataFrame({"x": [10, 20]}),
#     "Parts_tbl": pd.DataFrame({"part": ["p1", "p2"]}),
# }

# # Use a path you KNOW exists & can write to
# base_path = "./data"   # <-- safer than "/data/" on many systems

# save_data_locally(data, base_path)

In [48]:

def data_loading_from_local(base_path, server_name):
    # full_path = os.path.join(base_path, server_name)
    RO_file_name = os.path.join(base_path, server_name, "RO_tbl.csv")
    Req_file_name = os.path.join(base_path, server_name, "Req_tbl.csv")
    Labor_file_name = os.path.join(base_path, server_name, "Labor_tbl.csv")
    Parts_file_name = os.path.join(base_path, server_name, "Parts_tbl.csv")

    RO_df = pd.read_csv(RO_file_name)
    Req_df = pd.read_csv(Req_file_name)
    Labor_df = pd.read_csv(Labor_file_name)
    Parts_df = pd.read_csv(Parts_file_name)

    return RO_df, Req_df, Labor_df, Parts_df
 

In [49]:
def refactor_description_column(df, description_col="fldPartDesc"):
    
    # this function will select all the rows where column fldPartMasterRef = 1 and then for those rows, it will replace the value in fldDescription column with the value in fldPartDesc column. However, the value in fldPartDesc column should be not empty, otherwise the value in fldDescription column will remain unchanged. Before replacing the value, make a copy of the original value in fldDescription column and store it in a new column called Original_fldDescription. aslo, add a column named changed_description which will be True if the description is changed and False if it is not changed.


    mask = (df["fldPartsMasterRef"] == 1) & (df["fldPartDesc"].notna()) & (df["fldPartDesc"] != "")
    df["Original_" + description_col] = df[description_col]
    df["changed_description"] = False
    df.loc[mask, description_col] = df.loc[mask, "fldPartDesc"]
    df.loc[mask, "changed_description"] = True
    return df



In [50]:
def check_part_status(PartNumber):
    # load the parts master data
    parts_master_df = pd.read_csv("./data/parts_mastertbl.csv")  # adjust path    

    # check if the PartNumber exists in the parts master data
    if PartNumber in parts_master_df["fldPartNumber"].values:
        return "Active"
    
     # if not found, return "Inactive"
    return "Inactive"



def get_part_status_on_db(db_server, partnumber):
    # pull data for 

    server_conn = SERVER_conn(db_server)

    query_ = f"SELECT * FROM Parts_tblPartsCurrent WITH(NOLOCK) WHERE fldPartNumber = '{partnumber}'"
    result = db_request(query_, server_conn)

    if result is None:
        return "Database error"
    
    if isinstance(result, pd.DataFrame) and result.empty:
        return "Part not found"
    
    return result


def get_status_for_list_of_parts_on_db(db_server, partnumbers_list):
    # pull data for 

    server_conn = SERVER_conn(db_server)

    # Convert partnumbers_list to a string representation for SQL query
    partnumbers_str = ", ".join([f"'{part}'" for part in partnumbers_list if not pd.isna(part)])

    query_ = f"SELECT * FROM Parts_tblPartsCurrent WITH(NOLOCK) WHERE fldPartNumber IN ({partnumbers_str})"
    result = db_request(query_, server_conn)

    if result is None:
        return "Database error"
    
    if isinstance(result, pd.DataFrame) and result.empty:
        return "Part not found"
    
    return result[['fldPartNumber', 'fldPartDescription', 'fldStatus', 'fldClassRef','fldManufacturerRef', 'fldPartsMasterRef']]
    

    


def load_new_parts_to_localPartsMaster(df):
    db_path = "./data/parts_mastertbl.csv"

    start = datetime.now()
    print(f"[load_new_parts] Started : {start:%Y-%m-%d %H:%M:%S}")

    # Ensure master file exists
    if not os.path.exists(db_path):
        empty_df = pd.DataFrame(columns=[
            "fldPartNumber", "fldPartDescription", "fldStatus",
            "fldManufacturerRef", "fldPartsMasterRef"
        ])
        empty_df.to_csv(db_path, index=False)

    # Load master
    parts_master_df = pd.read_csv(db_path)

    # Clean column names
    df.columns = df.columns.str.strip()
    parts_master_df.columns = parts_master_df.columns.str.strip()

    # Make sure status is comparable (string)
    df["fldStatus"] = df["fldStatus"].astype(str).str.strip()

    # Identify new parts: not in master AND status in [1,2,4]
    new_parts = df[
        ~df["fldPartNumber"].isin(parts_master_df["fldPartNumber"]) &
        df["fldStatus"].isin(['1', '2', '4'])
    ].copy()

    print(f"[load_new_parts] New parts found: {len(new_parts)}")

    if not new_parts.empty:
        # Keep only the columns we care about
        new_parts_df = new_parts[[
            "fldPartNumber",
            "fldPartDescription",
            "fldStatus",
            "fldManufacturerRef",
            "fldPartsMasterRef"
        ]].copy()

        parts_master_df = pd.concat([parts_master_df, new_parts_df], ignore_index=True)
        parts_master_df.to_csv(db_path, index=False)
        print(f"[load_new_parts] Master rows after append: {len(parts_master_df)}")
    else:
        print("[load_new_parts] No new parts to add.")

    end = datetime.now()
    print(f"[load_new_parts] Finished: {end:%Y-%m-%d %H:%M:%S}")
    print(f"[load_new_parts] Time taken: {end - start}")




def update_existing_parts_in_localPartsMaster(df):
    db_path = "./data/parts_mastertbl.csv"

    start = datetime.now()
    print(f"[update_parts] Started : {start:%Y-%m-%d %H:%M:%S}")

    # Ensure the master file exists
    if not os.path.exists(db_path):
        empty_df = pd.DataFrame(columns=[
            "fldPartNumber", "fldPartDescription", "fldStatus",
            "fldManufacturerRef", "fldPartsMasterRef"
        ])
        empty_df.to_csv(db_path, index=False)

    # Load master
    parts_master_df = pd.read_csv(db_path)

    # Normalize column names
    df.columns = df.columns.str.strip()
    parts_master_df.columns = parts_master_df.columns.str.strip()

    # Keep only rows that have a non-null description (your business rule)
    df = df[df["fldPartDescription"].notna()].copy()

    # If df is empty after filtering, bail early
    if df.empty:
        print("[update_parts] No rows with non-null descriptions; nothing to update.")
        end = datetime.now()
        print(f"[update_parts] Finished: {end:%Y-%m-%d %H:%M:%S}")
        print(f"[update_parts] Time taken: {end - start}")
        return

    # ---- Handle duplicates in the input safely ----
    # If there are multiple rows per fldPartNumber, pick the last non-null per field
    update_fields = ["fldPartDescription", "fldStatus", "fldManufacturerRef", "fldPartsMasterRef"]

    # Sort if you have a notion of "latest"; if not, this still works
    df = df.sort_values("fldPartNumber")

    # For each part, keep the last row (you can change to first if needed)
    df_updates = df.drop_duplicates(subset=["fldPartNumber"], keep="last")

    # We only need fldPartNumber + fields to update
    cols_for_merge = ["fldPartNumber"] + [f for f in update_fields if f in df_updates.columns]
    df_updates = df_updates[cols_for_merge]

    # ---- Merge master with updates on fldPartNumber ----
    merged = parts_master_df.merge(
        df_updates,
        on="fldPartNumber",
        how="left",
        suffixes=("", "_new")
    )

    # ---- Vectorized conditional update per field ----
    for field in update_fields:
        new_col = field + "_new"
        if new_col in merged.columns:
            # Update only when new value is not null and not ""
            mask = merged[new_col].notna() & (merged[new_col] != "")
            if mask.any():
                merged.loc[mask, field] = merged.loc[mask, new_col]

    # Drop the *_new helper columns
    new_cols = [c for c in merged.columns if c.endswith("_new")]
    merged.drop(columns=new_cols, inplace=True)

    # Save back to CSV
    merged.to_csv(db_path, index=False)

    end = datetime.now()
    print(f"[update_parts] Finished: {end:%Y-%m-%d %H:%M:%S}")
    print(f"[update_parts] Time taken: {end - start}")

In [51]:
# modelName = "Escape"
# Servers = ["DB_server_130",'DB_server_172']

In [ ]:
# this is the main execution block where we pull data for both sites, run analysis, and save results locally. You can modify the modelName and Servers list as needed. 

modelName = "Escape"
Servers = ["DB_server_130",'DB_server_172', 'DB_server_114', 'DB_server_129', 'DB_server_168']

RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_pull(modelName, Servers[0])
RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_pull(modelName, Servers[1])
RO_tbl_114, request_tbl_114, labourline_tbl_114, partslines_tbl_114 = data_pull(modelName, Servers[2])
RO_tbl_129, request_tbl_129, labourline_tbl_129, partslines_tbl_129 = data_pull(modelName, Servers[3])
RO_tbl_168, request_tbl_168, labourline_tbl_168, partslines_tbl_168 = data_pull(modelName, Servers[4])



dataset_info(RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130, "130")
print("---------------------------------------------------------------------")
dataset_info(RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172, "172")
dataset_info(RO_tbl_114, request_tbl_114, labourline_tbl_114, partslines_tbl_114, "114")
dataset_info(RO_tbl_114, request_tbl_114, labourline_tbl_114, partslines_tbl_114, "129")
dataset_info(RO_tbl_114, request_tbl_114, labourline_tbl_114, partslines_tbl_114, "168")





In [271]:
all_data_dict = {"server_130": {
                "RO_tbl": RO_tbl_130,
                "Req_tbl": request_tbl_130, 
                "Labor_tbl":labourline_tbl_130, 
                "Parts_tbl": partslines_tbl_130},
        "server_172": {
                "RO_tbl": RO_tbl_172,
                "Req_tbl": request_tbl_172, 
                "Labor_tbl":labourline_tbl_172, 
                "Parts_tbl": partslines_tbl_172},
        "server_114": {
                "RO_tbl": RO_tbl_114,
                "Req_tbl": request_tbl_114, 
                "Labor_tbl":labourline_tbl_114, 
                "Parts_tbl": partslines_tbl_114},
        "server_129": {
                "RO_tbl": RO_tbl_129,
                "Req_tbl": request_tbl_129, 
                "Labor_tbl":labourline_tbl_129, 
                "Parts_tbl": partslines_tbl_129},
        "server_168": {
                "RO_tbl": RO_tbl_168,
                "Req_tbl": request_tbl_168, 
                "Labor_tbl":labourline_tbl_168, 
                "Parts_tbl": partslines_tbl_168}
}




In [ ]:
base_path = "./data"   # the base dir for data

for key, data in all_data_dict.items():
     save_data_locally(data, base_path, key)

In [277]:
all_data_dict["server_130"]["Req_tbl"]

,fldId,fldWorkItemRef,fldSequence,fldDescription,fldRequestCodeRef,fldRequestCode,fldRequestedTime,fldOrderNumber,fldLastUpdated
0,A9DDD156-B8A6-4A17-ADEB-E6826B5ADECC,3AE76151-5182-4B64-83A5-0009E346029F,1,INSPECTION,A37F4713-441E-44FF-9244-CEB45C8AF90C,90FCZI,0.9,1,2025-07-31 12:04:44.540
1,E2527E82-2EEE-4731-8095-0A19ADDA657C,3AE76151-5182-4B64-83A5-0009E346029F,2,ETCH\r\n,639DC2FA-5A5D-4AAB-BC50-DFD624560EE0,90FCZETCH,0.3,1,2025-07-31 12:04:44.540
2,6E73B5F3-6D25-441F-988B-9082646E2C4D,2CF360A9-94CF-45A1-B58A-0277A895959C,1,USED CAR INSPECTION PACKAGE\r\n- USED CAR INSP...,97B07344-2A46-4F3E-8EE0-2BE2269C0AB7,92FCZUCI,1.0,1,2024-02-13 07:53:28.217
3,A35E1A2C-55AB-44AA-B72D-C84293A23667,2CF360A9-94CF-45A1-B58A-0277A895959C,2,USED CAR INSPECTION PACKAGE\r\n- CHECK GLASS\r\n,97B07344-2A46-4F3E-8EE0-2BE2269C0AB7,92FCZUCI,0.0,1,2024-02-13 07:53:28.217
4,C578B8B2-86F9-47FA-A7E3-65F36010F6D8,2CF360A9-94CF-45A1-B58A-0277A895959C,3,Tricare Ultimate Protect - Tricare Certified V...,8286B9F3-63EC-458F-914D-B8464B3D8708,TUP,0.0,1,2024-02-13 07:53:28.220
...,...,...,...,...,...,...,...,...,...
207002,63698312-EA3E-403E-A970-ECDE18A2A9E3,922DD5E8-6210-414D-AFD2-FFF778E00B26,3,TIRE MEASUREMENTS,EDF1E18B-A057-4015-850B-8969A7E22AB4,12FCZTIRES,0.0,0,2017-08-11 00:00:00.000
207003,B433900F-1298-48D8-BA4F-BFBE6C1EBC43,922DD5E8-6210-414D-AFD2-FFF778E00B26,4,REAR PASS SEATBELT WILL NOT EXTEND,2FD5A91A-59AA-40C8-AF69-5E5735CDBD18,11FCZRS,0.0,0,2017-08-11 00:00:00.000
207004,9E3583B3-F416-49F0-962A-83E910928EBF,922DD5E8-6210-414D-AFD2-FFF778E00B26,5,NaN,C3E376AE-A60C-4EB6-BB8F-CCA11C6B9584,01FCZNM,0.0,0,2017-08-11 00:00:00.000
207005,A2DE73DA-832D-4C4F-941A-7B75B42F2C8E,922DD5E8-6210-414D-AFD2-FFF778E00B26,6,COMPLIMENTARY WHEEL ALIGNMENT CHECK,E8CED6D5-A6F1-4D3D-B641-D2595BF963CC,99FCZWAC,0.0,0,2017-08-11 00:00:00.000


In [ ]:

# loading data sets
base_path = "./data"


RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_loading_from_local(base_path, "server_130")
RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_loading_from_local(base_path, "server_172")
RO_tbl_114, request_tbl_114, labourline_tbl_114, partslines_tbl_114 = data_loading_from_local(base_path, "server_114")
RO_tbl_129, request_tbl_129, labourline_tbl_129, partslines_tbl_129 = data_loading_from_local(base_path, "server_129")
RO_tbl_168, request_tbl_168, labourline_tbl_168, partslines_tbl_168 = data_loading_from_local(base_path, "server_168")


partslines_tbl_130 = refactor_description_column(partslines_tbl_130)
partslines_tbl_172 = refactor_description_column(partslines_tbl_172)
partslines_tbl_114 = refactor_description_column(partslines_tbl_114)
partslines_tbl_129 = refactor_description_column(partslines_tbl_129)
partslines_tbl_168 = refactor_description_column(partslines_tbl_168)





In [66]:

# # Load new parts Data
# load_new_parts_to_localPartsMaster(partslines_tbl_130)
# load_new_parts_to_localPartsMaster(partslines_tbl_172)
# load_new_parts_to_localPartsMaster(partslines_tbl_114)
# load_new_parts_to_localPartsMaster(partslines_tbl_129)
# load_new_parts_to_localPartsMaster(partslines_tbl_168)

# # Update the master part DB

# update_existing_parts_in_localPartsMaster(partslines_tbl_130)
# update_existing_parts_in_localPartsMaster(partslines_tbl_172)
# update_existing_parts_in_localPartsMaster(partslines_tbl_114)
# update_existing_parts_in_localPartsMaster(partslines_tbl_129)
# update_existing_parts_in_localPartsMaster(partslines_tbl_168)

In [284]:
# jobs_list = ["Water Pump", "Power steering", "Catalytic Converter"]

# def run_model_all_sites(data_dict, jobs_list):
#     results_dict = {}

#     for service in jobs_list:
#         # print(service)
#         # execute model 
#         # loop through all sites
#         for site_name in data_dict.keys(): 
            
#             results_dict[service][site_name] = execute_model(request_tbl_df = data_dict[site_name]["Req_tbl"], search_key_words = service, labour_line_df = data_dict[site_name]["Labor_tbl"], parts_line_df = data_dict[site_name]["Parts_tbl"], similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)

# run_model_all_sites(all_data_dict, jobs_list)


In [256]:
 
key_wrds = ["Water Pump", "Power steering", "Catalytic Converter"]

key_wrd = key_wrds[0]

results130_water_pump = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results172_water_pump = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results114_water_pump = execute_model(request_tbl_df = request_tbl_114, search_key_words = key_wrd, labour_line_df = labourline_tbl_114, parts_line_df = partslines_tbl_114, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results129_water_pump = execute_model(request_tbl_df = request_tbl_129, search_key_words = key_wrd, labour_line_df = labourline_tbl_129, parts_line_df = partslines_tbl_129, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results168_water_pump = execute_model(request_tbl_df = request_tbl_168, search_key_words = key_wrd, labour_line_df = labourline_tbl_168, parts_line_df = partslines_tbl_168, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
 

In [252]:

data = {
    "130": results130_water_pump,
    "174": results172_water_pump,
    "114": results114_water_pump,
    "129": results129_water_pump,
    "168": results168_water_pump
}

compare_recommended_partLists_side_by_side(data)
 

,Part,Avg_Freq_%,Recommended_PartNums,OEM_Part,non_OEM_Parts,Common_Parts_Across_All,130_frequency_%,130_PartNumbers,174_frequency_%,174_PartNumbers,114_frequency_%,114_PartNumbers,129_frequency_%,129_PartNumbers,168_frequency_%,168_PartNumbers
1,WATER PUMP (KIT/ASY),94.3,"[4S4Z 8501 E, 7S7Z 8501 C, DS7Z 8501 E, PW 493...","PW 493, PW 545, PW 556, PW 579, PW 624, PW 625...","4S4Z 8501 E, 7S7Z 8501 C, DS7Z 8501 E",None,95.69,"[PW 556, PW 625, PW 579, PW 545, PW 493, PW 68...",100.00,"[DS7Z 8501 E, 7S7Z 8501 C, 4S4Z 8501 E]",100.0,"[DS7Z 8501 E, 7S7Z 8501 C]",81.54,"[PW 493, PW 545, PW 556, PW 624, PW 625, PW 579]",94.44,"[7S7Z 8501 C, DS7Z 8501 E]"
2,GASKET - WATER PUMP,86.2,"[1S7Z 8507 AE, 6M5Z 6B752 A, AM5Z 9450 A, BE8Z...","1S7Z 8507 AE, 6M5Z 6B752 A, AM5Z 9450 A, BE8Z ...",None,{BE8Z 8507 A},79.14,"[BE8Z 8507 A, 1S7Z 8507 AE, AM5Z 9450 A, CV6Z ...",85.72,"[BE8Z 8507 A, BM5Z 6584 B]",100.0,"[BE8Z 8507 A, FL 910S]",66.16,"[BE8Z 8507 A, 1S7Z 8507 AE, 6M5Z 6B752 A, DS7Z...",100.00,[BE8Z 8507 A]
3,ANTI-FREEZE,55.5,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, VC 13 G, VC 1...","CVC 13 DLG, CVC 13 G, CVC 7 B2, VC 13 G, VC 7 B",VC 13DL G,None,30.94,"[CVC 7 B2, CVC 13 G, CVC 13 DLG]",24.68,"[VC 13 G, VC 7 B]",80.0,[VC 13 G],75.38,"[CVC 13 G, CVC 7 B2]",66.67,[VC 13DL G]
4,MC YELLOW COOLANT,46.8,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2]","CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2",None,"{CVC 7 D1LB2, CVC 13 G, CVC 13 DLG, CVC 7 B2}",46.76,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2]",NaN,[],NaN,[],NaN,[],NaN,[]
5,SCREW,31.2,"[W500015 S437, W713043 S437]","W500015 S437, W713043 S437",None,{W713043 S437},17.99,"[W713043 S437, W500015 S437]",NaN,[],NaN,[],NaN,[],44.44,[W713043 S437]
6,NUT - LOCKING,31.1,[W520102 S442],W520102 S442,None,{W520102 S442},22.30,[W520102 S442],NaN,[],NaN,[],43.08,[W520102 S442],27.78,[W520102 S442]
7,RETAINER - BEARING,28.1,[YS4Z 3N324 AA],YS4Z 3N324 AA,None,{YS4Z 3N324 AA},23.02,[YS4Z 3N324 AA],NaN,[],20.0,[YS4Z 3N324 AA],41.54,[YS4Z 3N324 AA],27.78,[YS4Z 3N324 AA]
8,SCREW AND WASHER ASY,27.8,[W716136 S442],W716136 S442,None,{W716136 S442},33.10,[W716136 S442],NaN,[],20.0,[W716136 S442],24.62,[W716136 S442],33.33,[W716136 S442]
9,NUT,27.5,"[W520213 S440, W520214 S440, W520214 S441, W52...","W520213 S440, W520214 S440, W520214 S441, W520...",None,{W520415 S442},33.09,"[W520214 S440, W520415 S442, W715135 S440]",NaN,[],20.0,"[W520415 S442, W520214 S441]",29.23,"[W520213 S440, W520214 S440, W520214 S450B, W5...",27.78,"[W520415 S442, W715135 S440, W520214 S440]"
10,V-BELT,26.6,"[F1EZ 8620 A, JK3 211 A, JK6 441, JK6 455 D, J...","F1EZ 8620 A, JK3 211 A, JK6 441, JK6 455 D, JK...",None,None,28.78,"[JK6 441, JK3 211 A, JK6 617 A]",NaN,[],20.0,[JK6 441],35.38,"[JK3 211 A, JK6 441, JK6 455 D, JK6 617 A]",22.22,[F1EZ 8620 A]


In [117]:
results168_water_pump.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Part             0 non-null      object
 1   frequency_%      0 non-null      object
 2   PartNumbers      0 non-null      object
 3   CombinedParts    0 non-null      object
 4   MatchedKeywords  0 non-null      object
dtypes: object(5)
memory usage: 132.0+ bytes


In [ ]:

# {'BE8Z 8507 A', 'BM5Z 6584 B'}{'W716136 S442'}['BE8Z 8507 A', '9L8Z 8507 A', '1S7Z 8507 AE', 'CV6Z 9450 E', 'AM5Z 9450 A', 'CV6Z 9450 D', 'BM5Z 9448 A', '36037', 'BM5Z 9439 A', 'BM5Z 6584 B']
compare_site_results_side_by_side(results130_water_pump, results172_water_pump, perc_thresh=8)
 

,Part,Recommended_PartNums,Common_Parts,PartNumbers_130,PartNumbers_172
0,WATER PUMP (KIT/ASY),"[4S4Z 8501 E, DS7Z 8501 E, PW 493, PW 545, PW ...",None,"[PW 556, PW 579, PW 625, PW 545, PW 493, PW 68...","[DS7Z 8501 E, 4S4Z 8501 E]"
1,GASKET - WATER PUMP,"[1S7Z 8507 AE, AM5Z 9450 A, BE8Z 8507 A, BM5Z ...","{BE8Z 8507 A, BM5Z 6584 B}","[BE8Z 8507 A, 1S7Z 8507 AE, CV6Z 9450 E, AM5Z ...","[BE8Z 8507 A, BM5Z 6584 B]"
2,MC YELLOW COOLANT,"[CVC 13 DLG, CVC 13 G, CVC 7 D1LB2]","{CVC 7 D1LB2, CVC 13 DLG, CVC 13 G}","[CVC 13 DLG, CVC 13 G, CVC 7 D1LB2]",[]
3,SCREW AND WASHER ASY,[W716136 S442],{W716136 S442},[W716136 S442],[W716136 S442]
4,NUT,"[W520214 S440, W520415 S442, W715135 S440, W71...","{W520415 S442, W520214 S440}","[W520415 S442, W520214 S440, W715135 S440]","[W715618 S437, W520214 S440, W520415 S442]"
5,ANTI-FREEZE,"[CVC 13 DLG, CVC 13 G, VC 13 G, VC 7 B]",None,"[CVC 13 G, CVC 13 DLG]","[VC 13 G, VC 7 B]"
6,V-BELT,"[F1EZ 8620 A, JK3 211 A, JK6 441, JK6 617 A]","{JK6 617 A, JK3 211 A}","[JK6 441, JK3 211 A, JK6 617 A]","[JK6 617 A, F1EZ 8620 A, JK3 211 A]"
7,RETAINER - BEARING,[YS4Z 3N324 AA],{YS4Z 3N324 AA},[YS4Z 3N324 AA],[YS4Z 3N324 AA]
8,NUT - LOCKING,[W520102 S442],{W520102 S442},[W520102 S442],[W520102 S442]
9,BOLT,"[5F9Z 4682 AA, 7N5Z 00812 A, BE8Z 6A340 A, W71...","{BE8Z 6A340 A, W716075 S442, W718250 S439, 5F9...","[W716075 S442, 5F9Z 4682 AA, BE8Z 6A340 A, W71...","[W712146 S437, W718250 S439, 5F9Z 4682 AA, 7N5..."


In [48]:

# parts_test = ['DS7Z 8501 E', '7S7Z 8501 C', '4S4Z 8501 AA', '4S4Z 8501 E', '9L8Z 8501 A', '4S4Z 8501 D']
parts_test = ['PW 556', 'PW 579', 'PW 625', 'PW 545', 'PW 493', 'PW 686', 'PW 624', 'PW 578', 'PW 481', 'PW 500', 'PW 570', 'PW 447', 'PW 584', 'WP2249', 'ZLWP2249', '58-670']
filter_existing_parts(parts_test)

['PW 556', 'PW 579', 'PW 625', 'PW 545', 'PW 493', 'PW 686', 'PW 624']

In [49]:
key_wrd = key_wrds[1]
results130_power_str = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_power_str = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,SENSOR - STEER,44.12,[CL8Z 3F818 A],[SENSOR - STEER],"{'include_any': [], 'exclude_any': []}"
1,BOLT,39.71,"[W712250 S437, W713065 S439]",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,BULK MERCON V,10.29,[CXT-5-L],[BULK MERCON V],"{'include_any': [], 'exclude_any': []}"
3,WATER PUMP (KIT/ASY),8.82,"[STP 152, STP 182, ZLCRD21-5271]","[PUMP ASY - POW, STEERING PUMP]","{'include_any': ['WATER', 'PUMP'], 'exclude_an..."
4,GEAR ASY - STE,5.88,"[STE 419, STE 175, STE 98, STE 282]",[GEAR ASY - STE],"{'include_any': [], 'exclude_any': []}"
5,V-BELT,4.41,"[JK6 844 C, JK6 931 AA, JK6 455 C]",[V-BELT],"{'include_any': [], 'exclude_any': []}"
6,SENSOR - STEERING RO,4.41,[CL8Z 3F818 A],[SENSOR - STEERING RO],"{'include_any': [], 'exclude_any': []}"
7,LINK,2.94,"[MEF 166, MEF 200]",[LINK],"{'include_any': [], 'exclude_any': []}"
8,SEAL,2.94,[388898 S],[SEAL],"{'include_any': [], 'exclude_any': []}"
9,LUBE KIT,2.94,[PKFL-500],[LUBE KIT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,SENSOR - STEERING ROTATION,38.76,[CL8Z 3F818 A],[SENSOR - STEERING ROTATION],"{'include_any': [], 'exclude_any': []}"
1,BOLT,33.33,"[W714807 S900, W716075 S442, 7N5Z 00812 A, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,COLUMN ASY - STEERING,14.73,"[CL8Z 3C529 D, PZ1Z 3C529 AQ, CL8Z 3C529 B]",[COLUMN ASY - STEERING],"{'include_any': [], 'exclude_any': []}"
3,GEAR ASY - STEERING,13.18,"[CV6Z 3504 WE, CV6Z 3504 AGE, CV6Z 3504 EE, CV...",[GEAR ASY - STEERING],"{'include_any': [], 'exclude_any': []}"
4,BOLT - HEX.HEAD,7.75,[W711137 S442],[BOLT - HEX.HEAD],"{'include_any': [], 'exclude_any': []}"
5,NUT - HEX.,7.75,"[W520203 S442, W520215 S440, W520113 S442, W70...",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
6,NPN PART UCS HISTORY,6.98,[NPN],[NPN PART UCS HISTORY],"{'include_any': [], 'exclude_any': []}"
7,COLUMN ASY - STEERIN,6.98,[CL8Z 3C529 C],[COLUMN ASY - STEERIN],"{'include_any': [], 'exclude_any': []}"
8,NUT,6.20,"[W715135 S440, W520214 S443, W520415 S442]",[NUT],"{'include_any': [], 'exclude_any': []}"
9,ALTERNATOR ASY,4.65,"[CJ5Z 10346 D, CJ5Z 10346 C, CJ5Z 10346 A, CJ5...",[ALTERNATOR ASY],"{'include_any': [], 'exclude_any': []}"


In [50]:

compare_site_results_side_by_side(results130_power_str, results172_power_str, perc_thresh=2)
 

,Part,Recommended_PartNums,Common_Parts,PartNumbers_130,PartNumbers_172
0,STEERING TORQUE SENSOR,[CL8Z 3F818 A],{CL8Z 3F818 A},[CL8Z 3F818 A],[]
1,BOLT,"[7N5Z 00812 A, W302562 S300, W712250 S437, W71...","{W713065 S439, W712250 S437}","[W712250 S437, W713065 S439]","[W714807 S900, W716075 S442, 7N5Z 00812 A, W71..."
2,BULK MERCON V,None,None,[],[]
3,WATER PUMP (KIT/ASY),None,None,[],[]
4,GEAR ASY - STE,None,None,[],[]
5,V-BELT,"[F1EZ 8620 A, JK6 844 C]",None,[JK6 844 C],[F1EZ 8620 A]
6,BATTERY,"[BAGM 48H6 760, BXT 96 R590, BXT 96R 590]",None,[BXT 96 R590],"[BXT 96R 590, BAGM 48H6 760]"
7,SEAL,"[388898 S, 9L8Z 1177 F]",None,[388898 S],[9L8Z 1177 F]
8,NUT - HEX.,"[W520113 S442, W520203 S442, W520215 S440, W70...",{W520113 S442},"[W701267 S440, W520113 S442]","[W520203 S442, W520215 S440, W520113 S442, W70..."
9,MERCON V ATF (1L),[CXT 5 LM6],{CXT 5 LM6},[CXT 5 LM6],[]


In [51]:

key_wrd = key_wrds[2]
results130_catal = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_catal = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,CATALYTIC CONVERTER (ALL),81.82,"[CV6Z 5E212 D, CV6Z 5E212 F, LX6Z 5E212 KZ, JJ...","[CONVERTER ASY, REAR CONVERTER]","{'include_any': ['CATALYTIC', 'CAT CONVERTER',..."
1,BOLT,63.64,"[5F9Z 4682 AA, W715681 S900, W716075 S442, W50...",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,NUT - HEX.,63.64,"[W520103 S403, W520203 S442, W520103 S442]",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
3,GASKET - WATER PUMP,45.45,"[BB5Z 6L612 A, CV6Z 9450 D, CV6Z 9450 E, AM5Z ...","[GASKET, GASKET - EXHAUST MAN]","{'include_any': ['GASKET'], 'exclude_any': []}"
4,CLAMP - EXHAUST,45.45,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",[CLAMP - EXHAUST],"{'include_any': [], 'exclude_any': []}"
5,BOLT AND WASHER ASY,18.18,"[W711806 S442, W709601 S442]",[BOLT AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"
6,NUT,18.18,"[W520415 S442, W716271 S437]",[NUT],"{'include_any': [], 'exclude_any': []}"
7,SEAL - REFER TO (PK-CN1Z) **,18.18,[CN1Z 7H424 B],[SEAL - REFER TO (PK-CN1Z) **],"{'include_any': [], 'exclude_any': []}"
8,SEAL,18.18,[CV6Z 7086 B],[SEAL],"{'include_any': [], 'exclude_any': []}"
9,RETAINER - BEARING,18.18,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords


In [52]:

compare_site_results_side_by_side(results130_catal, results172_catal, perc_thresh=1)
 

,Part,Recommended_PartNums,Common_Parts,PartNumbers_130,PartNumbers_172
0,CATALYTIC CONVERTER (ALL),[LX6Z 5E212 KZ],{LX6Z 5E212 KZ},[LX6Z 5E212 KZ],[]
1,BOLT,"[5F9Z 4682 AA, W500233 S442, W500635 S439, W71...","{W715681 S900, W500233 S442, W718250 S439, W50...","[5F9Z 4682 AA, W715681 S900, W716075 S442, W50...",[]
2,NUT - HEX.,"[W520103 S403, W520103 S442, W520203 S442]","{W520103 S403, W520203 S442, W520103 S442}","[W520103 S403, W520203 S442, W520103 S442]",[]
3,CLAMP - EXHAUST,"[JX6Z 5A215 C, LX6Z 5A215 A, LX6Z 5A215 D]","{LX6Z 5A215 A, JX6Z 5A215 C, LX6Z 5A215 D}","[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",[]
4,GASKET - WATER PUMP,"[AM5Z 9450 A, BB5Z 6L612 A, BM5Z 9450 A, CV6Z ...","{CV6Z 9450 E, BM5Z 9450 A, BB5Z 6L612 A, AM5Z ...","[BB5Z 6L612 A, CV6Z 9450 E, AM5Z 9450 A, BM5Z ...",[]
5,BOLT AND WASHER ASY,"[W709601 S442, W711806 S442]","{W711806 S442, W709601 S442}","[W711806 S442, W709601 S442]",[]
6,NUT,"[W520415 S442, W716271 S437]","{W716271 S437, W520415 S442}","[W520415 S442, W716271 S437]",[]
7,SEAL - REFER TO (PK-CN1Z) **,[CN1Z 7H424 B],{CN1Z 7H424 B},[CN1Z 7H424 B],[]
8,SEAL,[CV6Z 7086 B],{CV6Z 7086 B},[CV6Z 7086 B],[]
9,RETAINER - BEARING,[YS4Z 3N324 AA],{YS4Z 3N324 AA},[YS4Z 3N324 AA],[]


In [53]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
water_pump_keywrds = ["Water|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER"] 


In [54]:

all_parts_dfs_combined = results172_catal.copy()

all_parts_dfs_combined = pd.concat(
    [partslines_tbl_172, partslines_tbl_130])


In [55]:
def group_dfs(*dfs):
    return pd.concat(dfs, ignore_index=True)


In [56]:

def search_parts_by_keyword(df, search__keys):
    result = df[(df["fldPartDesc"].str.contains(search__keys[0], case= False, na=False)) &
                   (~df["fldPartDesc"].str.contains(search__keys[1], case= False, na=False))]
    display(result["fldPartDesc"].unique().tolist())
 

In [57]:
# "gasket", "coolant", "Nut", "Anti|Freeze",
key_words_for_water_pump_parts = ["belt", "nut", "bolt", "screw", "retainer", "seal", "oil", "stud"]


# Nuts, Seal, Screw, Retainer, stud

# key_words_for_catal_parts = ["converter", "bolt", 'nut', "gasket", "clamp", "washer", "seal", "retainer", "coolant", "hanger", "tube", "sensor", "insulator"]

for part_key_wrd in key_words_for_water_pump_parts:
    display(results172_water_pump[results172_water_pump["Part"].str.contains(part_key_wrd, case=False)])

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
3,V-BELT,7.79,"[JK6 617 A, F1EZ 8620 A, JK6 867 A, JK3 211 A]",[V-BELT],"{'include_any': [], 'exclude_any': []}"
12,BELT - TIMING,2.60,[BE8Z 6268 C],[BELT - TIMING],"{'include_any': [], 'exclude_any': []}"
21,TENSIONER - TIMING BELT,1.30,[BM5Z 6K254 A],[TENSIONER - TIMING BELT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
4,NUT,6.49,"[W715618 S437, W520214 S440, W520415 S442, W52...",[NUT],"{'include_any': [], 'exclude_any': []}"
5,NUT - LOCKING,5.19,[W520102 S442],[NUT - LOCKING],"{'include_any': [], 'exclude_any': []}"
13,RETAINER - NUT,2.60,[CCPZ 3B477 G],[RETAINER - NUT],"{'include_any': [], 'exclude_any': []}"
26,NUT - HEX.,1.30,[W520203 S442],[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
9,BOLT,3.9,"[W712146 S437, W718250 S439, 5F9Z 4682 AA, 7N5...",[BOLT],"{'include_any': [], 'exclude_any': []}"
15,BOLT - HEX.HEAD,1.3,[BE8Z 6379 AB],[BOLT - HEX.HEAD],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
7,SCREW AND WASHER ASY,5.19,[W716136 S442],[SCREW AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
6,RETAINER - BEARING,5.19,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"
13,RETAINER - NUT,2.60,[CCPZ 3B477 G],[RETAINER - NUT],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
17,SEAL,1.3,[9L8Z 1177 C],[SEAL],"{'include_any': [], 'exclude_any': []}"
18,SEALANT - SILICONE,1.3,[TA 357],[SEALANT - SILICONE],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
8,FILTER ASY - OIL,3.9,[BE8Z 6731 AC],[FILTER ASY - OIL],"{'include_any': [], 'exclude_any': []}"
19,SEPARATOR ASY - OIL,1.3,[DS7Z 6A785 C],[SEPARATOR ASY - OIL],"{'include_any': [], 'exclude_any': []}"
23,OIL - AUTOMATIC TRANSMISSION,1.3,[XT 10 QLVC],[OIL - AUTOMATIC TRANSMISSION],"{'include_any': [], 'exclude_any': []}"
24,OIL COOLER ASY,1.3,[DS7Z 6B856 A],[OIL COOLER ASY],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords


In [58]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
# water_pump_keywrds = ["Water Pump|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER|Coupling|Steering|coolant"] 

water_pump_gasket_keywrds = ["gasket - water", "None"] 

# water_pump_keywrds
combo_df = group_dfs(partslines_tbl_130, partslines_tbl_172)

search_parts_by_keyword(combo_df, water_pump_gasket_keywrds)


['GASKET - WATER', 'GASKET - WATER PUMP']

In [59]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
power_steering_keywrds = ["steering", "pump"] 

search_parts_by_keyword(combo_df, power_steering_keywrds)


['WHEEL ASY - STEERING',
 'SENSOR - STEERING RO',
 'STEERING TORQUE SENSOR',
 'LOCK ASY - STEERING',
 'GEAR ASY - STEERING',
 'used steering column',
 'CORE - GEAR ASY - STEERING',
 'SWITCH ASY - STEERING WHEEL',
 'COLUMN ASY - STEERING',
 'P&A STEERING SHAFT BOLT',
 'SENSOR ASY - STEERING ROTATION',
 'SENSOR - STEERING ROTATION',
 'LOCK ASY - STEERING AND IGNITI']